# Build the verified recursion-to-iteration dataset on a T4

Same script, same gate, same inputs as the local run. Only the generation
backend changes: `--backend hf` samples a whole function's attempts in one
batch on the GPU instead of one call at a time on the CPU.

| | CPU (llama-server) | T4 (batched) |
|---|---:|---:|
| per function, 6 samples | ~120 s | a few seconds |
| 582 functions | ~19 hours | ~1-2 hours |

**The GPU does not improve the yield.** Same weights, same answers. It makes
attempts cheap enough to afford more of them, and the failures are correlated -
a function the model will not de-recurse tends to stay that way - so more
samples help sublinearly. Plan on 15-25%, not 80%.

Once generation is fast, **compiling and running the candidates becomes the
bottleneck**, and that is CPU work Colab gives you two cores for. That is the
real limit on this notebook, not the model.

Runtime → Change runtime type → **T4 GPU** before running anything.

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!g++ --version | head -1   # the gate needs this; Colab already has it

/bin/bash: line 1: nvidia-smi: command not found
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0


## 1. The code

Cloned rather than pasted, so the gate that decides which rows exist is the
same one with tests behind it. A private repo needs a token with `repo` scope.

In [3]:
import os
from getpass import getpass

REPO = "safi892/fyp_training"
BRANCH = "language"

token = getpass("GitHub token (blank if the repo is public): ").strip()
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

!rm -rf /content/fyp && git clone -q -b {BRANCH} {url} /content/fyp
os.chdir("/content/fyp")
!git log --oneline -1

c9627f0 (HEAD -> language, origin/language) Upload and download instead of Drive, now the VS Code extension does both


In [4]:
# Kept narrow on purpose: Colab's preinstalled torch is fine and reinstalling it
# costs several minutes and sometimes a restart.
!pip install -q transformers peft accelerate
!pip list 2>/dev/null | grep -E "^(torch|transformers|peft|accelerate) "

accelerate                            1.14.0
peft                                  0.20.0
torch                                 2.11.0+cpu
transformers                          5.15.0


## 2. The inputs, without uploading anything

Neither input has to come from the laptop.

**The functions** ship with the clone. `cleaned/merged_cleaned.jsonl` is 40 MB
and git-ignored, but the notebook never needed the corpus - only the 582
recursive functions whose signature the driver can drive. Those are extracted
once on the laptop and committed as `inputs.jsonl`, 246 KB, so `git clone`
brings them.

**The weights** download from Hugging Face. The default below is the *base*
model, `Qwen/Qwen2.5-Coder-1.5B-Instruct`, which needs no adapter and no upload.

Using the base model is not only a convenience, and it is worth reading before
assuming the fine-tune would do better. **83% of the fine-tune's own optimize
targets left the recursion in place.** It was trained on examples that tidied
recursive code rather than removing the recursion, and it reproduces that: the
CPU run measured 8%, with ten of eleven rejections never attempting the
transformation at all.

So the base model is a real candidate to propose better rewrites here, and the
short run below is what decides it. If the adapter is wanted anyway, put it on
the runtime by any means and set `ADAPTER` to its path.

In [ ]:
import os

INPUTS = "/content/fyp/my_data_annotation/recursion_optimization/inputs.jsonl"
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER = None          # None = base model, no upload, nothing to fetch

assert os.path.exists(INPUTS), "the clone did not bring inputs.jsonl - check the branch"
count = sum(1 for line in open(INPUTS) if line.strip())
print(f"{count} recursive functions from the clone   (expected 582)")
print(f"proposer: {BASE}{' + ' + ADAPTER if ADAPTER else '  (base, no adapter)'}")

## 3. Check the inputs before spending GPU time

This should print the same counts as the laptop. If it does not, the corpus
that arrived is not the corpus the numbers were measured on, and nothing below
is comparable.

In [ ]:
import sys

sys.path.insert(0, "/content/fyp/src")
sys.path.insert(0, "/content/fyp/scripts")
from pathlib import Path

from build_optimize_dataset import WORDING, drivable_recursive, judge

functions = drivable_recursive(Path(INPUTS), 40)
print(f"{len(functions)} drivable recursive functions   (expected 582)")
print(f"wording: {WORDING[:60]}...")

# The gate itself, on a case whose answer is known. A wrong loop must be
# rejected here or every number this notebook produces is worthless.
rec = "int fact(int n){ if(n<=1) return 1; return n*fact(n-1); }"
itr = "int fact(int n){ int r=1; for(int i=2;i<=n;i++) r*=i; return r; }"
assert judge(rec, itr, 10.0) is None
assert judge(rec, itr.replace("r=1", "r=0"), 10.0) == "different output"
assert judge(rec, rec, 10.0) == "still recursive"
print("gate rejects a wrong rewrite and a non-rewrite: ok")

## 4. A short run first

Twenty functions, to get a real yield and a real per-function time before
committing the session. Sixteen samples rather than six, because on a GPU they
cost about the same as one.

In [ ]:
# /content, so the Contents view can reach it. Survives a kernel restart;
# does NOT survive the runtime being recycled, so download it once the short
# run has proved the yield rather than at the end of a two-hour session.
OUT = "/content/verified.jsonl"

# Built as a string rather than inlined into the ! magic: an expression with
# quotes in it does not survive IPython's {} interpolation reliably, and a
# broken command line here looks exactly like a broken model.
def build_cmd(limit, samples=16, temperature=0.9):
    adapter = f"--adapter {ADAPTER}" if ADAPTER else ""
    return (
        f"cd /content/fyp && python scripts/build_optimize_dataset.py "
        f"--backend hf --base {BASE} {adapter} "
        f"--corpus {INPUTS} --out {OUT} "
        f"--limit {limit} --samples {samples} --temperature {temperature}"
    )

print(build_cmd(20))

In [ ]:
!{build_cmd(20)}

**Read the yield before going on.** Rows kept ÷ 20 is what 582 will give you.
At 20% that is about 115 rows; at 8% it is 47 and the GPU has bought speed
rather than a dataset. Either is a result worth writing down - the second one
says the limit is the model and not the budget.

## 5. The rest

Resumable: it skips functions already kept and already failed, so a disconnect
costs only the function in flight. Re-run this cell after a reconnect.

In [ ]:
# The rest. Resumable: it skips functions already kept and already failed, so a
# disconnect costs only the function in flight. Re-run after a reconnect.
!{build_cmd(582)}

## 6. What came out

Re-verified here rather than trusted, because the rows were written by a
process that could have been interrupted mid-line.

In [ ]:
import json

rows = [json.loads(line) for line in open(OUT) if line.strip()]
print(f"{len(rows)} verified rows")

bad = [r for r in rows if judge(r["code"], r["improved_code"], 10.0) is not None]
print(f"{len(bad)} fail a second check   (must be 0)")

if rows:
    print("\n--- one row ---")
    print(rows[0]["code"][:300])
    print("    ->")
    print(rows[0]["improved_code"][:300])

## 7. Getting it onto the laptop

Three routes, cheapest first.

**Contents view** — the activity bar shows the runtime's `/content`. Right-click
`verified.jsonl`, download, and put it in
`my_data_annotation/recursion_optimization/`.

**Print it and copy** — the cell below writes the whole file into the notebook
output, which is saved in the `.ipynb` on the laptop. Ugly, survives a
disconnect, and needs no clicking. Fine at a hundred rows.

**Push it** — the runtime already has the repo cloned and a token if you used
one, so it can commit the file straight back to the branch. That is the only
route that leaves the result somewhere other than one machine.

The local CPU run writes the same format to the same filename, so the two merge
by concatenation. De-duplicate on `code`: both runs draw from the same 582
inputs and will overlap.

In [ ]:
# Route 2: the rows, in the notebook output, saved with the .ipynb on the laptop.
rows = [json.loads(line) for line in open(OUT) if line.strip()]
print(f"# {len(rows)} verified rows - paste into my_data_annotation/recursion_optimization/verified.jsonl")
for row in rows:
    print(json.dumps(row, ensure_ascii=False))

Or route 3, straight back to the branch. Only worth it if the token you used to
clone had write access.

In [ ]:
# Route 3: commit the result to the branch from the runtime.
DEST = "my_data_annotation/recursion_optimization/verified.colab.jsonl"

!cp {OUT} /content/fyp/{DEST}
!cd /content/fyp && git config user.email "colab@local" && git config user.name "colab" \
  && git add {DEST} \
  && git commit -q -m "Verified recursion-to-iteration rows built on a Colab T4" \
  && git push -q origin {BRANCH} && echo "pushed {DEST}"